In [ ]:
import os
import sys
import numpy as np
import traceback
from datetime import datetime
import psycopg2
from psycopg2.extras import Json
import pywt

# 프로젝트 루트 경로 추가
sys.path.append('../')
from core.config import ConfigLoader

# DB 연결 정보
db_cfg = ConfigLoader(config_path="../core/config.yaml").db
print("[DB CONFIG]", db_cfg)

conn = psycopg2.connect(
    host=db_cfg['host'], 
    port=db_cfg['port'], 
    user=db_cfg['user'], 
    password=db_cfg['password'], 
    dbname=db_cfg['dbname']
)
conn.autocommit = True
cur = conn.cursor()
print("DB 연결 완료")


In [ ]:
# wavelet_vector 테이블 생성
create_table_sql = '''
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS wavelet_vector (
    id serial PRIMARY KEY,
    origin_vector_id integer NOT NULL REFERENCES origin_vector(id),
    wavelet_family varchar(32) NOT NULL,
    level integer NOT NULL,
    mode varchar(16) NOT NULL DEFAULT 'low',
    original_dim integer NOT NULL,
    compressed_dim integer NOT NULL,
    compression_ratio float,
    parameters json,
    embedding vector,
    created_at timestamp,
    log text
);

-- 인덱스 생성
CREATE INDEX IF NOT EXISTS idx_wavelet_vector_origin_id ON wavelet_vector(origin_vector_id);
CREATE INDEX IF NOT EXISTS idx_wavelet_vector_family_level ON wavelet_vector(wavelet_family, level);
CREATE INDEX IF NOT EXISTS idx_wavelet_vector_mode ON wavelet_vector(mode);
'''

try:
    cur.execute(create_table_sql)
    print('wavelet_vector 테이블 및 인덱스 생성 완료')
except Exception as e:
    print(f'테이블 생성 에러: {e}')


In [ ]:
# Wavelet 변환 설정 (routes/extract_router.py와 동일)
wavelet_families = [
    'haar', 'db2', 'db4', 'sym2', 'sym4', 'coif1', 'bior1.3', 'rbio1.3'
]

# 512차원 벡터의 최대 decomposition level 계산
test_vector = np.random.randn(512)
max_levels = []

for wname in wavelet_families:
    try:
        max_level = pywt.dwt_max_level(len(test_vector), pywt.Wavelet(wname).dec_len)
        max_levels.append(max_level)
        print(f"{wname}: max_level = {max_level}")
    except Exception as e:
        max_levels.append(1)
        print(f"{wname}: 에러 - {e}, max_level = 1로 설정")

max_level_all = max(max_levels)
print(f"\\n전체 최대 레벨: {max_level_all}")
print(f"Wavelet families: {wavelet_families}")
print(f"각 family별 max levels: {max_levels}")


In [ ]:
# origin_vector에서 데이터 로드
cur.execute("""
    SELECT id, embedding, image_path, label
    FROM origin_vector 
    WHERE log LIKE 'Face detected%'  -- 얼굴이 정상 검출된 경우만
    ORDER BY id
""")

origin_vectors = cur.fetchall()
print(f"처리할 origin_vector 수: {len(origin_vectors)}")

if len(origin_vectors) > 0:
    print("샘플 데이터:")
    sample = origin_vectors[0]
    print(f"  ID: {sample[0]}")
    print(f"  이미지 경로: {sample[2]}")
    print(f"  라벨: {sample[3]}")
    print(f"  임베딩 차원: {len(sample[1])}")
    
    # 예상 총 작업량 계산
    total_combinations = sum(max_levels) * 2  # 각 family별 level 수 × 2 modes
    total_tasks = len(origin_vectors) * total_combinations
    print(f"\\n예상 총 작업량:")
    print(f"  origin_vectors: {len(origin_vectors)}")
    print(f"  families × levels × modes: {len(wavelet_families)} families × levels × 2 modes = {total_combinations}")
    print(f"  총 Wavelet 변환 작업: {total_tasks}")
else:
    print("처리할 데이터가 없습니다!")


In [ ]:
# 모든 wavelet family, level, mode 조합으로 변환 및 저장
modes = ['low', 'high']  # DCT와 동일한 모드
processed_count = 0
error_count = 0
skipped_count = 0

print("Wavelet 변환 및 저장 시작...")

for idx, (origin_id, embedding_list, image_path, label) in enumerate(origin_vectors):
    # 임베딩을 numpy 배열로 변환
    embedding_vector = np.array(embedding_list, dtype=np.float32)
    
    # 모든 wavelet family에 대해 처리
    for fam_idx, wavelet_name in enumerate(wavelet_families):
        max_level = max_levels[fam_idx]
        
        # 각 level에 대해 처리
        for level in range(1, max_level + 1):
            # 각 mode에 대해 처리
            for mode in modes:
                try:
                    # 중복 방지: 이미 처리된 조합인지 확인
                    cur.execute("""
                        SELECT id FROM wavelet_vector 
                        WHERE origin_vector_id=%s AND wavelet_family=%s AND level=%s AND mode=%s
                    """, (origin_id, wavelet_name, level, mode))
                    
                    if cur.fetchone():
                        skipped_count += 1
                        continue
                    
                    # Wavelet 변환 수행
                    compressed_vec, original_dim, compressed_dim, compression_ratio, log_msg = wavelet_transform_extract(
                        embedding_vector, wavelet_name, level, mode
                    )
                    
                    if compressed_vec is not None:
                        # 파라미터 정보
                        parameters = {
                            'wavelet_family': wavelet_name,
                            'level': level,
                            'mode': mode,
                            'original_image_path': image_path,
                            'original_label': label
                        }
                        
                        # DB 저장
                        cur.execute("""
                            INSERT INTO wavelet_vector 
                            (origin_vector_id, wavelet_family, level, mode, original_dim, 
                             compressed_dim, compression_ratio, parameters, embedding, created_at, log)
                            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                        """, (
                            origin_id, wavelet_name, level, mode, original_dim,
                            compressed_dim, compression_ratio, Json(parameters), 
                            compressed_vec.tolist(), datetime.now(), log_msg
                        ))
                        
                        processed_count += 1
                    else:
                        # 에러 발생 시 로그만 저장 (embedding은 빈 배열)
                        cur.execute("""
                            INSERT INTO wavelet_vector 
                            (origin_vector_id, wavelet_family, level, mode, original_dim, 
                             compressed_dim, compression_ratio, parameters, embedding, created_at, log)
                            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                        """, (
                            origin_id, wavelet_name, level, mode, original_dim,
                            0, 0.0, Json({'wavelet_family': wavelet_name, 'level': level, 'mode': mode}), 
                            [], datetime.now(), log_msg
                        ))
                        error_count += 1
                        
                except Exception as e:
                    error_count += 1
                    print(f"에러 [origin_id:{origin_id}, {wavelet_name}, level:{level}, mode:{mode}]: {str(e)}")
                    
                    # 에러 로그 저장
                    try:
                        cur.execute("""
                            INSERT INTO wavelet_vector 
                            (origin_vector_id, wavelet_family, level, mode, original_dim, 
                             compressed_dim, compression_ratio, parameters, embedding, created_at, log)
                            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                        """, (
                            origin_id, wavelet_name, level, mode, len(embedding_vector),
                            0, 0.0, Json({'wavelet_family': wavelet_name, 'level': level, 'mode': mode}), 
                            [], datetime.now(), f"EXCEPTION: {str(e)}"
                        ))
                    except:
                        pass  # 중복 등으로 인한 DB 에러는 무시
    
    # 주기적 진행 상황 출력
    if idx % 100 == 0:
        print(f'진행률: {idx+1}/{len(origin_vectors)} (처리:{processed_count}, 에러:{error_count}, 스킵:{skipped_count})')

print(f'\\nWavelet 변환 완료!')
print(f'- 총 origin_vector: {len(origin_vectors)}')
print(f'- 처리 완료: {processed_count}')
print(f'- 에러: {error_count}')
print(f'- 스킵(중복): {skipped_count}')


In [ ]:
# 결과 확인 및 통계
print("=== Wavelet 변환 결과 확인 ===")

# 전체 변환된 레코드 수
cur.execute("SELECT COUNT(*) FROM wavelet_vector")
total_count = cur.fetchone()[0]
print(f"총 변환된 레코드 수: {total_count}")

# Wavelet family별 통계
cur.execute("""
    SELECT wavelet_family, COUNT(*) as count, 
           AVG(compression_ratio) as avg_ratio,
           MIN(compressed_dim) as min_dim,
           MAX(compressed_dim) as max_dim
    FROM wavelet_vector 
    WHERE log LIKE 'SUCCESS%'
    GROUP BY wavelet_family 
    ORDER BY count DESC
""")
print("\\nWavelet Family별 통계 (성공한 변환만):")
for row in cur.fetchall():
    family, count, avg_ratio, min_dim, max_dim = row
    print(f"  {family}: {count}개, 평균압축비={avg_ratio:.3f}, 차원범위={min_dim}-{max_dim}")

# Level별 통계
cur.execute("""
    SELECT level, COUNT(*) as count,
           AVG(compression_ratio) as avg_ratio,
           AVG(compressed_dim) as avg_dim
    FROM wavelet_vector 
    WHERE log LIKE 'SUCCESS%'
    GROUP BY level 
    ORDER BY level
""")
print("\\nLevel별 통계 (성공한 변환만):")
for row in cur.fetchall():
    level, count, avg_ratio, avg_dim = row
    print(f"  Level {level}: {count}개, 평균압축비={avg_ratio:.3f}, 평균차원={avg_dim:.1f}")

# Mode별 통계
cur.execute("""
    SELECT mode, COUNT(*) as count,
           AVG(compression_ratio) as avg_ratio,
           AVG(compressed_dim) as avg_dim
    FROM wavelet_vector 
    WHERE log LIKE 'SUCCESS%'
    GROUP BY mode 
    ORDER BY mode
""")
print("\\nMode별 통계 (성공한 변환만):")
for row in cur.fetchall():
    mode, count, avg_ratio, avg_dim = row
    print(f"  {mode}: {count}개, 평균압축비={avg_ratio:.3f}, 평균차원={avg_dim:.1f}")

# 성공/실패 통계
cur.execute("""
    SELECT 
        CASE 
            WHEN log LIKE 'SUCCESS%' THEN 'Success'
            WHEN log LIKE 'ERROR%' THEN 'Error'
            ELSE 'Exception'
        END as status,
        COUNT(*) as count
    FROM wavelet_vector 
    GROUP BY 
        CASE 
            WHEN log LIKE 'SUCCESS%' THEN 'Success'
            WHEN log LIKE 'ERROR%' THEN 'Error'
            ELSE 'Exception'
        END
""")
print("\\n변환 결과 통계:")
for status, count in cur.fetchall():
    print(f"  {status}: {count}개")

# 샘플 데이터 조회 (각 mode별로)
for mode in ['low', 'high']:
    cur.execute("""
        SELECT wv.wavelet_family, wv.level, wv.compressed_dim, wv.compression_ratio, 
               ov.image_path, ov.label
        FROM wavelet_vector wv
        JOIN origin_vector ov ON wv.origin_vector_id = ov.id
        WHERE wv.mode = %s AND wv.log LIKE 'SUCCESS%'
        ORDER BY wv.wavelet_family, wv.level, wv.origin_vector_id
        LIMIT 3
    """, (mode,))
    
    print(f"\\n샘플 데이터 ({mode} mode):")
    for row in cur.fetchall():
        family, level, compressed_dim, ratio, image_path, label = row
        print(f"  {family}-L{level}, 압축차원={compressed_dim}, 압축비={ratio:.3f}, {label}/{image_path}")

print("\\n=== 작업 완료 ===")


In [ ]:
# DB 연결 종료
cur.close()
conn.close()
print("DB 연결 종료")


In [ ]:
# Wavelet 변환 함수 정의 (DCT 방식과 동일하게 수정)
def wavelet_transform_extract(vector, wavelet_name='haar', level=1, mode='low'):
    """
    1차원 벡터에서 Wavelet 변환 후 저주파/고주파 성분 추출
    DCT와 동일한 방식으로 keep_dim 개념 적용
    """
    try:
        # 1D wavelet decomposition
        coeffs = pywt.wavedec(vector, wavelet_name, level=level)
        
        # 모든 계수를 하나의 벡터로 concatenate
        all_coeffs = np.concatenate(coeffs)
        original_dim = len(vector)
        
        # 첫 번째 계수(저주파)의 크기를 keep_dim으로 사용
        keep_dim = len(coeffs[0])
        
        if mode == 'low':
            # 저주파 성분만 유지 (처음 keep_dim개만 유지)
            compressed_vector = all_coeffs[:keep_dim]
            compressed_dim = keep_dim
            
        elif mode == 'high':
            # 고주파 성분만 유지 (처음 keep_dim개를 0으로, 나머지 유지)
            compressed_vector = all_coeffs[keep_dim:]
            compressed_dim = len(compressed_vector)
            
        else:
            raise ValueError("mode는 'low' 또는 'high'만 허용")
        
        compression_ratio = compressed_dim / original_dim
        log_msg = f"SUCCESS: wavelet={wavelet_name}, level={level}, mode={mode}, {original_dim}->{compressed_dim} (ratio={compression_ratio:.3f})"
        
        return compressed_vector, original_dim, compressed_dim, compression_ratio, log_msg
        
    except Exception as e:
        error_msg = f"ERROR: wavelet={wavelet_name}, level={level}, mode={mode}, error={str(e)}"
        return None, len(vector), 0, 0.0, error_msg

# 테스트 (두 모드 모두)
for mode in ['low', 'high']:
    test_result = wavelet_transform_extract(test_vector, 'haar', 3, mode)
    print(f"테스트 결과 ({mode}):", test_result[4])  # log 메시지만 출력
